In [5]:
import numpy as np
import random
from typing import Tuple, List, Dict

In [6]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import timm 
from sklearn.metrics import (
    accuracy_score, 
    roc_auc_score, 
    precision_recall_fscore_support, 
)
import torch.optim as optim

In [1]:
from utils.config import (
    MODEL_NAME, NUM_CLASSES, BATCH_SIZE, EVAL_BATCH_SIZE, NUM_WORKERS, DEVICE,
    LEARNING_RATE, WEIGHT_DECAY, NUM_EPOCHS, T_MAX_LR_SCHEDULER_EPOCHS, CHECKPOINT_PATH,
    TRAIN_DIR, TEST_DIR, VAL_DIR, CHECKPOINT_PATH_1, FREEZE_EPOCHS, FINE_TUNE_LR, PRECISION_BOOST_FACTOR
)

In [2]:
from utils.dataset import ChestXrayDataset, train_tf, val_tf, get_file_paths_and_labels

#### Set seeds for reproducibility

In [3]:
def set_seed(seed: int = 42) -> None:
    """Sets the seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        # For deterministic behavior
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False
    print(f"Seeds set to {seed}.")

#### -------------------- Model Definition --------------------

In [7]:
def get_model(model_name: str, num_classes: int, pretrained: bool = True, freeze_base: bool = False) -> nn.Module:
    """Instantiates a Vision Transformer (ViT) model from timm and optionally freezes its base."""
    
    model = timm.create_model(
        model_name, 
        pretrained=pretrained, 
        num_classes=num_classes
    )
    
    if freeze_base:
        # Freeze all parameters except those in the classification head ('head' in timm's ViT)
        for name, param in model.named_parameters():
             if 'head' not in name:
                 param.requires_grad = False
             
    print(f"Model: {model_name} instantiated. Number of classes: {num_classes}. Base Frozen: {freeze_base}")
    return model

#### -------------------- Training and Evaluation Functions --------------------
##### 1. Training

In [8]:
def train_epoch(
    model: nn.Module, 
    loader: DataLoader, 
    optimizer: torch.optim.Optimizer, 
    criterion: nn.Module, 
    device: str,
    scaler: torch.cuda.amp.GradScaler,
    scheduler: optim.lr_scheduler._LRScheduler = None 
) -> Tuple[float, float]:
    """Runs a single training epoch."""
    model.train()
    losses: List[float] = []
    all_preds: List[int] = []
    all_labels: List[int] = []
    
    # Use enumerate for tracking progress more clearly if needed, but tqdm is sufficient
    for images, labels in tqdm(loader, desc="Training"):
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        # Mixed Precision Training
        with torch.autocast(device_type=device, dtype=torch.float16):
            logits = model(images)
            loss = criterion(logits, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        if scheduler is not None:
             scheduler.step()
        
        # Metrics collection
        losses.append(loss.item())
        
        # Use .detach().cpu() only when necessary for non-gradient operations
        preds = torch.argmax(logits.detach().cpu(), dim=1).numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy()) # labels are already on device, move back for numpy
        
    acc = accuracy_score(all_labels, all_preds)
    return float(np.mean(losses)), float(acc)

##### 2. Evaluating

In [9]:
def eval_epoch(
    model: nn.Module, 
    loader: DataLoader, 
    criterion: nn.Module, 
    device: str
) -> Dict[str, float]:
    """Runs a single evaluation epoch and returns comprehensive metrics."""
    model.eval()
    losses: List[float] = []
    all_probs: List[float] = []
    all_preds: List[int] = []
    all_labels: List[int] = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating"):
            images = images.to(device)
            labels = labels.to(device)
            
            with torch.autocast(device_type=device, dtype=torch.float16):
                logits = model(images)
                loss = criterion(logits, labels)
                # Softmax to get probabilities for AUC, choosing the positive class (index 1)
                probs = torch.softmax(logits, dim=1)[:, 1] 
                
            losses.append(loss.item())
            
            # Metrics collection
            preds = torch.argmax(logits.cpu(), dim=1).numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
    # Calculate comprehensive metrics
    avg_loss = np.mean(losses)
    acc = accuracy_score(all_labels, all_preds)
    
    try:
        # AUC requires probabilities for the positive class
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        # Happens if only one class is present in the batch/dataset (rare, but good to handle)
        auc = 0.0
        
    # precision, recall, f1 for binary classification
    prec, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary', zero_division=0
    )
    
    # Optional: Confusion Matrix
    # cm = confusion_matrix(all_labels, all_preds)
    
    return {
        'loss': avg_loss,
        'accuracy': acc,
        'auc': auc,
        'precision': prec,
        'recall': recall,
        'f1': f1,
    }

### -------------------- Main Training Loop --------------------

##### 1. Data Preparation

In [10]:
set_seed(42)
try:
    train_files, train_labels, train_weights_np = get_file_paths_and_labels(TRAIN_DIR)
    # val_files, val_labels, _ = get_file_paths_and_labels(VAL_DIR)
    
    boosted_weights_np = train_weights_np.copy()
    boosted_weights_np[0] *= PRECISION_BOOST_FACTOR
    
    val_files, val_labels, _ = get_file_paths_and_labels(TEST_DIR)
    
except FileNotFoundError as e:
    print(f"Error: Data directory not found. Please update BASE_DIR in config.py.")
    print(f"Missing directory: {e}")

Seeds set to 42.
Loaded 5216 samples from C:\Users\HP\Desktop\SLIIT\Y4 SEM 1\DL\Ass\Assignment\DL-project\ViT\dataset\chest_xray\train. Class counts: {'NORMAL': 1341, 'PNEUMONIA': 3875}
Calculated class weights: [1.9448173  0.67303226] (Index 0: NORMAL, Index 1: PNEUMONIA)
Loaded 624 samples from C:\Users\HP\Desktop\SLIIT\Y4 SEM 1\DL\Ass\Assignment\DL-project\ViT\dataset\chest_xray\test. Class counts: {'NORMAL': 234, 'PNEUMONIA': 390}
Calculated class weights: [1.33333333 0.8       ] (Index 0: NORMAL, Index 1: PNEUMONIA)


##### 2. Datasets and DataLoaders

In [11]:
train_ds = ChestXrayDataset(train_files, train_labels, transform=train_tf)
val_ds   = ChestXrayDataset(val_files, val_labels, transform=val_tf)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)
val_loader   = DataLoader(
    val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)
print("DataLoaders initialized.")

DataLoaders initialized.


### 3. Training phases
##### Phase 1 

In [12]:
# --- Phase 1: Train ONLY the Head (for 5 epochs) ---
FULL_EPOCHS = NUM_EPOCHS - FREEZE_EPOCHS

# 1. Instantiate the model with the base FROZEN
print(f"\n--- Starting Phase 1: Frozen Base (Head Only) for {FREEZE_EPOCHS} Epochs ---")
model = get_model(MODEL_NAME, NUM_CLASSES, freeze_base=True).to(DEVICE)


--- Starting Phase 1: Frozen Base (Head Only) for 5 Epochs ---
Model: vit_base_patch16_224 instantiated. Number of classes: 2. Base Frozen: True


##### Configurations for phase 1 : Optimiser, Scaler, Weights, Scheduler

In [13]:
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), 
    lr=LEARNING_RATE, 
    weight_decay=WEIGHT_DECAY
)

scaler = torch.amp.GradScaler(device=DEVICE)  # Initialize scaler once

best_val_auc = 0.0 # Best AUC tracker

# class_weights = torch.tensor(train_weights_np, dtype=torch.float32).to(DEVICE) 
class_weights = torch.tensor(boosted_weights_np, dtype=torch.float32).to(DEVICE) 
criterion = nn.CrossEntropyLoss(weight=class_weights)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=T_MAX_LR_SCHEDULER_EPOCHS * len(train_loader) # T_max is number of steps
) 

In [14]:
print(f"\n--- Starting Phase 1: Frozen Base (Head Only) for {FREEZE_EPOCHS} Epochs ---")

for epoch in range(1, FREEZE_EPOCHS + 1):
    # TRAIN
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE, scaler)
        
    # VALIDATE
    val_metrics = eval_epoch(model, val_loader, criterion, DEVICE)
    val_loss, val_acc, val_auc, val_prec, val_rec, val_f1 = (
        val_metrics['loss'], val_metrics['accuracy'], val_metrics['auc'], 
        val_metrics['precision'], val_metrics['recall'], val_metrics['f1']
    )
        
    # SCHEDULER is not needed here
        
    # LOGGING
    print(
        f"Epoch {epoch}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, Val AUC: {val_auc:.4f}, "
        f"Val Recall: {val_rec:.4f}, Val F1: {val_f1:.4f}"
    )
        
    # SAVE BEST MODEL
    if val_metrics['auc'] > best_val_auc:
        best_val_auc = val_metrics['auc']
        torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'best_val_auc': best_val_auc}, CHECKPOINT_PATH_1)
        print(f"--- Model saved! New best AUC: {best_val_auc:.4f} ---")
        # Or CHECKPOINT_PATH as prev 



--- Starting Phase 1: Frozen Base (Head Only) for 5 Epochs ---


Validating: 100%|██████████| 20/20 [00:10<00:00,  1.94it/s]


Epoch 1/30 | Train Loss: 0.4569, Train Acc: 0.7743 | Val Loss: 0.5487, Val Acc: 0.8429, Val AUC: 0.9242, Val Recall: 0.9590, Val F1: 0.8842
--- Model saved! New best AUC: 0.9242 ---


Validating: 100%|██████████| 20/20 [00:03<00:00,  5.12it/s]


Epoch 2/30 | Train Loss: 0.2917, Train Acc: 0.8890 | Val Loss: 0.5090, Val Acc: 0.8622, Val AUC: 0.9278, Val Recall: 0.9590, Val F1: 0.8969
--- Model saved! New best AUC: 0.9278 ---


...

Epoch 3/30 | Train Loss: 0.2425, Train Acc: 0.9062 | Val Loss: 0.5263, Val Acc: 0.8670, Val AUC: 0.9307, Val Recall: 0.9667, Val F1: 0.9008
--- Model saved! New best AUC: 0.9307 ---


Validating: 100%|██████████| 20/20 [00:03<00:00,  5.19it/s]


Epoch 4/30 | Train Loss: 0.2135, Train Acc: 0.9153 | Val Loss: 0.6580, Val Acc: 0.8189, Val AUC: 0.9307, Val Recall: 0.9795, Val F1: 0.8712


Validating: 100%|██████████| 20/20 [00:03<00:00,  5.14it/s]

Epoch 5/30 | Train Loss: 0.1996, Train Acc: 0.9185 | Val Loss: 0.6409, Val Acc: 0.8317, Val AUC: 0.9273, Val Recall: 0.9744, Val F1: 0.8786


### Phase 2
##### Unfreezing and fine-tuning

In [15]:
# --- Phase 2: Unfreeze and Fine-Tune the Whole Model ---
print(f"\n--- Transitioning to Phase 2: Unfreezing All Layers ---")

# Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

# Re-define Optimizer with the new (lower) Fine-Tune LR for all parameters
optimizer = optim.AdamW(model.parameters(), lr=FINE_TUNE_LR, weight_decay=WEIGHT_DECAY)

# Use CosineAnnealingLR for smooth decay during fine-tuning
TOTAL_STEPS = FULL_EPOCHS * len(train_loader) 
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=TOTAL_STEPS
) 


--- Transitioning to Phase 2: Unfreezing All Layers ---


In [16]:
print(f"--- Starting Phase 2: Fine-Tuning All Layers for {FULL_EPOCHS} Epochs ---")

for epoch in range(FREEZE_EPOCHS + 1, NUM_EPOCHS + 1):
     # TRAIN
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE, scaler, scheduler)
        
    # VALIDATE
    val_metrics = eval_epoch(model, val_loader, criterion, DEVICE)
    val_loss, val_acc, val_auc, val_prec, val_rec, val_f1 = (
        val_metrics['loss'], val_metrics['accuracy'], val_metrics['auc'], 
        val_metrics['precision'], val_metrics['recall'], val_metrics['f1']
    )
        
    # SCHEDULER is called inside train epoch since we used TOTAL_STEPS for T_max (per batch) 
        
    # LOGGING
    print(
        f"Epoch {epoch}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, Val AUC: {val_auc:.4f}, "
        f"Val Recall: {val_rec:.4f}, Val F1: {val_f1:.4f}"
    )
        
    # SAVE BEST MODEL
    if val_metrics['auc'] > best_val_auc:
        best_val_auc = val_metrics['auc']
        torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'best_val_auc': best_val_auc}, CHECKPOINT_PATH_1)
        print(f"--- Model saved! New best AUC: {best_val_auc:.4f} ---")  
        # Or CHECKPOINT_PATH as prev 

--- Starting Phase 2: Fine-Tuning All Layers for 25 Epochs ---


Validating: 100%|██████████| 20/20 [00:03<00:00,  5.12it/s]


Epoch 6/30 | Train Loss: 0.1732, Train Acc: 0.9314 | Val Loss: 0.4700, Val Acc: 0.9038, Val AUC: 0.9670, Val Recall: 0.9821, Val F1: 0.9274
--- Model saved! New best AUC: 0.9670 ---


Validating: 100%|██████████| 20/20 [00:05<00:00,  3.95it/s]


Epoch 7/30 | Train Loss: 0.1004, Train Acc: 0.9580 | Val Loss: 1.5584, Val Acc: 0.7917, Val AUC: 0.9755, Val Recall: 1.0000, Val F1: 0.8571
--- Model saved! New best AUC: 0.9755 ---


...

Epoch 28/30 | Train Loss: 0.0043, Train Acc: 0.9987 | Val Loss: 3.1104, Val Acc: 0.7949, Val AUC: 0.9774, Val Recall: 1.0000, Val F1: 0.8590


Validating: 100%|██████████| 20/20 [00:04<00:00,  4.87it/s]


Epoch 29/30 | Train Loss: 0.0036, Train Acc: 0.9983 | Val Loss: 3.2589, Val Acc: 0.7869, Val AUC: 0.9759, Val Recall: 1.0000, Val F1: 0.8543


Validating: 100%|██████████| 20/20 [00:04<00:00,  4.85it/s]

Epoch 30/30 | Train Loss: 0.0030, Train Acc: 0.9987 | Val Loss: 3.3242, Val Acc: 0.7837, Val AUC: 0.9759, Val Recall: 1.0000, Val F1: 0.8525
